**Subscriber Stength Analysis**



1.   Do larger platforms actually retain users better?
2.   Which platform has the highest subscriber lifetime value?

1.   Is growth being driven by retention or replacement?
2.   Are high-subscriber platforms masking weak retention?




In [1]:
#IMPORT package

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#UPLOAD reference file

from google.colab import files
uploaded = files.upload()

Saving SVOD_Platform_Count.csv to SVOD_Platform_Count.csv


In [3]:
#LOAD file

df = pd.read_csv('SVOD_Platform_Count.csv')
df.head()

,Platform,Subscribers (Estimate)
0,Netflix,"325,000,000"
1,Amazon Prime,"200,000,000"
2,Disney+,"131,600,000"
3,"HBO Maxincluding HBO, Discovery+ and other WBD","131,600,000"
4,Tencent Video,"114,000,000"


In [4]:
#CLEAN dataset


# Rename column
df = df.rename(columns={"Subscribers (Estimate)": "Subscribers"})

# Remove commas and convert to numeric
df["Subscribers"] = df["Subscribers"].replace(",", "", regex=True).astype(float)

# Clean platform names
df["Platform"] = df["Platform"].replace({
    "Amazon Prime": "Amazon Prime Video",
    "Apple TV": "Apple TV+",
    "HBO Maxincluding HBO, Discovery+ and other WBD": "HBO Max"
})

# View cleaned data
df.head()

,Platform,Subscribers
0,Netflix,325000000.0
1,Amazon Prime Video,200000000.0
2,Disney+,131600000.0
3,HBO Max,131600000.0
4,Tencent Video,114000000.0


In [5]:
#BUILD ARPU - Average revenue per user - data

# Add ARPU (monthly, in USD)
arpu_values = {
    "Netflix": 15,
    "Amazon Prime Video": 12,
    "Disney+": 10,
    "HBO Max": 14,
    "Tencent Video": 8,
    "iQIYI": 7,
    "JioHotstar": 5,
    "Paramount+": 9,
    "Hulu": 13,
    "Peacock": 8,
    "Hotstar": 5,
    "SonyLiv": 6,
    "Apple TV+": 11,
    "iFlix": 4,
    "CuriosityStream": 3,
    "ESPN": 12
}

df["ARPU"] = df["Platform"].map(arpu_values)

df.head()

,Platform,Subscribers,ARPU
0,Netflix,325000000.0,15.0
1,Amazon Prime Video,200000000.0,12.0
2,Disney+,131600000.0,10.0
3,HBO Max,131600000.0,14.0
4,Tencent Video,114000000.0,8.0


In [6]:
#CHECK if ARPU was asigned to all platforms in dataset

df[df["ARPU"].isna()]

,Platform,Subscribers,ARPU
16,StarTimes,20000000.0,NaN
17,DAZN,20000000.0,NaN
18,Crunchyroll,17000000.0,NaN
19,Viu,13800000.0,NaN
20,Starz,12200000.0,NaN
...,...,...,...
86,Blim TV,297000.0,NaN
87,Sweet TV,150000.0,NaN
88,Foxtel,131000.0,NaN
89,Volia TV,120000.0,NaN


In [7]:
#CREATE platform tiers to assign ARPU for all platforms

def assign_arpu(platform):
    platform = platform.lower()

    # Premium global platforms
    if any(x in platform for x in ["netflix", "hbo", "hulu", "disney", "prime", "apple", "paramount"]):
        return 12

    # Mid-tier / mixed monetization
    elif any(x in platform for x in ["peacock", "espn", "sony", "viu", "wetv"]):
        return 8

    # Regional / emerging markets
    elif any(x in platform for x in ["iqiyi", "tencent", "hotstar", "jio", "iflix"]):
        return 5

    # Niche / smaller platforms
    else:
        return 6

In [8]:
#APPLY tiers to data

df["ARPU"] = df["Platform"].apply(assign_arpu)
df.head()

,Platform,Subscribers,ARPU
0,Netflix,325000000.0,12
1,Amazon Prime Video,200000000.0,12
2,Disney+,131600000.0,12
3,HBO Max,131600000.0,12
4,Tencent Video,114000000.0,5


In [9]:
#CHECK for NaN values - no NaN returned

df["ARPU"].isna().sum()

np.int64(0)

In [11]:
#CREATE more variation in the ARPU numbers

import numpy as np

def assign_arpu(platform):
    if platform == "Netflix":
        return np.random.uniform(13, 18)
    elif platform == "Amazon Prime Video":
        return np.random.uniform(10, 14)
    elif platform == "Disney+":
        return np.random.uniform(8, 12)
    elif platform == "HBO Max":
        return np.random.uniform(11, 15)
    elif platform == "Tencent Video":
        return np.random.uniform(3, 6)
    else:
        return np.random.uniform(7, 12)


In [12]:
#CHECK

df["ARPU"] = df["Platform"].apply(assign_arpu)

In [13]:
#ROUND, display to check

df["ARPU"] = df["ARPU"].round(2)
df.head()

,Platform,Subscribers,ARPU
0,Netflix,325000000.0,14.70
1,Amazon Prime Video,200000000.0,12.55
2,Disney+,131600000.0,9.97
3,HBO Max,131600000.0,11.45
4,Tencent Video,114000000.0,5.72


In [17]:
#BUILD annual revenue data

# Calculate annual revenue
df["Annual_Revenue"] = df["Subscribers"] * df["ARPU"] * 12

# Format for readability (optional but helpful)
df["Annual_Revenue_Billions"] = df["Annual_Revenue"] / 1e9

df.head(20)

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions,Churn_Rate
0,Netflix,325000000.0,14.70,5.733000e+10,57.330000,0.025
1,Amazon Prime Video,200000000.0,12.55,3.012000e+10,30.120000,0.030
2,Disney+,131600000.0,9.97,1.574462e+10,15.744624,0.035
3,HBO Max,131600000.0,11.45,1.808184e+10,18.081840,0.033
4,Tencent Video,114000000.0,5.72,7.824960e+09,7.824960,0.060
5,iQIYI,101100000.0,10.45,1.267794e+10,12.677940,0.060
6,JioHotstar,100000000.0,8.15,9.780000e+09,9.780000,0.060
7,Paramount+,78900000.0,8.14,7.706952e+09,7.706952,0.045
8,Hulu,64100000.0,9.08,6.984336e+09,6.984336,0.032
9,Peacock,44000000.0,8.05,4.250400e+09,4.250400,0.050


In [19]:
#CREATE churn rate

def refine_churn(platform):
    platform = platform.lower()

    if "netflix" in platform:
        return 0.025
    elif "disney" in platform:
        return 0.035
    elif "hbo" in platform:
        return 0.033
    elif "prime" in platform:
        return 0.03
    elif "apple" in platform:
        return 0.04
    elif "hulu" in platform:
        return 0.032
    elif "paramount" in platform:
        return 0.045
    elif any(x in platform for x in ["iqiyi", "tencent", "hotstar", "jio", "iflix"]):
        return 0.06
    else:
        return 0.05

df["Churn_Rate"] = df["Platform"].apply(refine_churn)

df.head(20)


,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions,Churn_Rate
0,Netflix,325000000.0,14.70,5.733000e+10,57.330000,0.025
1,Amazon Prime Video,200000000.0,12.55,3.012000e+10,30.120000,0.030
2,Disney+,131600000.0,9.97,1.574462e+10,15.744624,0.035
3,HBO Max,131600000.0,11.45,1.808184e+10,18.081840,0.033
4,Tencent Video,114000000.0,5.72,7.824960e+09,7.824960,0.060
5,iQIYI,101100000.0,10.45,1.267794e+10,12.677940,0.060
6,JioHotstar,100000000.0,8.15,9.780000e+09,9.780000,0.060
7,Paramount+,78900000.0,8.14,7.706952e+09,7.706952,0.045
8,Hulu,64100000.0,9.08,6.984336e+09,6.984336,0.032
9,Peacock,44000000.0,8.05,4.250400e+09,4.250400,0.050


In [20]:
#ADD more variation in the Churn rate for better analysis - generate custom logic.

import numpy as np

def refine_churn(platform):
    platform = platform.lower()

    if "netflix" in platform:
        base = 0.025
        noise = np.random.uniform(-0.004, 0.004)

    elif "disney" in platform:
        base = 0.035
        noise = np.random.uniform(-0.005, 0.005)

    elif "hbo" in platform:
        base = 0.033
        noise = np.random.uniform(-0.004, 0.004)

    elif "prime" in platform:
        base = 0.03
        noise = np.random.uniform(-0.004, 0.004)

    elif "apple" in platform:
        base = 0.04
        noise = np.random.uniform(-0.006, 0.006)

    elif "hulu" in platform:
        base = 0.032
        noise = np.random.uniform(-0.004, 0.004)

    elif "paramount" in platform:
        base = 0.045
        noise = np.random.uniform(-0.006, 0.006)

    elif any(x in platform for x in ["iqiyi", "tencent", "hotstar", "jio", "iflix"]):
        base = 0.06
        noise = np.random.uniform(-0.008, 0.008)

    else:
        base = 0.05
        noise = np.random.uniform(-0.005, 0.005)

    churn = base + noise

    # guardrails (important)
    return max(0.01, min(churn, 0.08))



In [24]:
#APPLY & store new churn rates rounded to 4 places

df["Churn_Rate"] = df["Platform"].apply(refine_churn)
df["Churn_Rate"] = df["Churn_Rate"].round(4)

df.head()

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions,Churn_Rate
0,Netflix,325000000.0,14.70,5.733000e+10,57.330000,0.0213
1,Amazon Prime Video,200000000.0,12.55,3.012000e+10,30.120000,0.0302
2,Disney+,131600000.0,9.97,1.574462e+10,15.744624,0.0332
3,HBO Max,131600000.0,11.45,1.808184e+10,18.081840,0.0343
4,Tencent Video,114000000.0,5.72,7.824960e+09,7.824960,0.0535


In [25]:
#CALCULATE churn - Lifetime of subscriber, revenue over lifetime
#ADD column - Lifetime Months - mhow long subscribers stay on the platform
#ADD column - LTV - Lifetime value of subscribers

# Calculate subscriber lifetime in months
df["Lifetime_Months"] = 1 / df["Churn_Rate"]

# Calculate lifetime value
df["LTV"] = df["ARPU"] * df["Lifetime_Months"]

df.head()

,Platform,Subscribers,ARPU,Annual_Revenue,Annual_Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,14.70,5.733000e+10,57.330000,0.0213,46.948357,690.140845
1,Amazon Prime Video,200000000.0,12.55,3.012000e+10,30.120000,0.0302,33.112583,415.562914
2,Disney+,131600000.0,9.97,1.574462e+10,15.744624,0.0332,30.120482,300.301205
3,HBO Max,131600000.0,11.45,1.808184e+10,18.081840,0.0343,29.154519,333.819242
4,Tencent Video,114000000.0,5.72,7.824960e+09,7.824960,0.0535,18.691589,106.915888


In [26]:
#REMOVE annual revenue duplicate column

df = df.drop(columns=["Annual_Revenue"])
df.head()

,Platform,Subscribers,ARPU,Annual_Revenue_Billions,Churn_Rate,Lifetime_Months,LTV
0,Netflix,325000000.0,14.70,57.330000,0.0213,46.948357,690.140845
1,Amazon Prime Video,200000000.0,12.55,30.120000,0.0302,33.112583,415.562914
2,Disney+,131600000.0,9.97,15.744624,0.0332,30.120482,300.301205
3,HBO Max,131600000.0,11.45,18.081840,0.0343,29.154519,333.819242
4,Tencent Video,114000000.0,5.72,7.824960,0.0535,18.691589,106.915888


In [29]:
#CHECK for NaN values in dataset

df.isna().sum()

,0
Platform,0
Subscribers,0
ARPU,0
Annual_Revenue_Billions,0
Churn_Rate,0
Lifetime_Months,0
LTV,0


In [31]:
# Use True/False to check, confirm dataset is clean

df.isna().any().any()

np.False_

In [33]:
#EXPORT new dataset for visualization in Tableau

df.to_csv("subscriber_strength_model_v2.csv", index=False)

from google.colab import files
files.download("subscriber_strength_model_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>